In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
clusters = pd.read_csv(f'{projdir}/csv/clusters/rna_leiden_v7.csv', sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

### start from raw

In [ ]:
igvf = sc.read_h5ad(f'{projdir}/h5ad/rna/liu/igvf_intersect_atac.h5ad')

In [ ]:
igvf.obs['cluster'] = clusters.reindex(igvf.obs.index)['coarse_v7']

In [ ]:
sc.pp.normalize_total(igvf, target_sum=1e4)

In [ ]:
sc.pp.log1p(igvf)

In [ ]:
liu = sc.read_h5ad(f'{projdir}/h5ad/rna/liu/seurat_reindexed.h5ad')

In [ ]:
liu.obs['coarse'] = [x[:2] for x in liu.obs['cluster']]
liu.obs['coarse'].unique()

In [ ]:
### already transformed
liu.X[:100,:100].toarray()

### why don't I just do a naive correlation?

In [ ]:
liu_clusters = liu.obs['coarse'].unique()
liu_results = []
for s in liu_clusters:
    clust_adata = liu[liu.obs['coarse'] == s]
    clust_mean = np.array(clust_adata.X.mean(axis=0))
    liu_results.append(clust_mean)

In [ ]:
igvf_clusters = igvf.obs['cluster'].unique()
igvf_results = []
for s in igvf_clusters:
    clust_igvf = igvf[igvf.obs['cluster'] == s]
    clust_mean = np.array(clust_igvf.X.mean(axis=0))
    igvf_results.append(clust_mean)

In [ ]:
liu_merged = np.vstack(liu_results)
igvf_merged = np.vstack(igvf_results)
liu_merged.shape, igvf_merged.shape

In [ ]:
merged_pseudo = pd.DataFrame(np.vstack([liu_merged, igvf_merged]))
merged_pseudo.index = np.hstack([liu_clusters, igvf_clusters])
merged_pseudo.columns = igvf.var.index
merged_pseudo.shape

In [ ]:
merged_pseudo.head()

In [ ]:
corr_df = merged_pseudo.T.corr()
corr_df.shape

In [ ]:
corr_df.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 8))

ax = sns.heatmap(corr_df, cmap='coolwarm', vmin=-1, vmax=1)

In [ ]:
corr_df.index

In [ ]:
select = ['fm', 'ni',  'pr', 're'] + tmp_order
to_plot = corr_df.loc[select, select].iloc[:4,:]

In [ ]:
liu_dict = {'fm' : 'Fibroblast', 'ni' : 'Novel intermediate',
 'pr' : 'Primed iPSC', 're' : 'Non-reprogrammed'}
to_plot.index = [liu_dict.get(x) for x in to_plot.index]

In [ ]:
to_plot

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

ax = sns.heatmap(to_plot, cmap='coolwarm', vmin=0.5, vmax=1)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
to_plot.to_csv(f'{outdir}/figure1/liu_igvf_corr.csv', sep='\t')

In [ ]:
!date